In [6]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


### Part 1.1 — Your first API call

In [7]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content


answer = ask_llm("What is a large language model in one sentence?")
print(answer)

A large language model is a type of artificial intelligence (AI) designed to process and understand human language, using complex algorithms and massive amounts of data to generate human-like text, answer questions, and complete tasks.


**Student Reasoning — Anatomy of a call**

The system role gives the model overall instructions about how it should behave, while the user role contains the specific request or question from the user.
A token is roughly a small unit of text. API providers bill per token rather than per request because different requests can vary greatly in the amount of text processed and generated. Charging by tokens therefore reflects the amount of computational work used more accurately.

### Part 1.2 — Temperature: the randomness dial

In [8]:
question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature = 0.0")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"{i+1}. {answer}")

print("\nTemperature = 1.2")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"{i+1}. {answer}")

Temperature = 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes the idea of t

**Student Reasoning — Temperature**

At temperature 0.0, the model was more consistent across the five runs. Several names appeared repeatedly, especially “Makola Save,” “Trader’s Treasure,” “Market Mobi,” and “Kokroko Savings.” The wording still changed slightly, but the model kept returning to similar ideas. At temperature 1.2, the responses were noticeably more varied and creative, with names such as “Accra Hustle Account,” “TradoSave,” “Kelewele Savings,” “AfroSave,” “MarketMax Savings,” and “Kokoza Savings.”
For a loan decision-support system, I would use a low temperature, close to 0.0, because loan decisions require consistency, reliability, and predictable reasoning. A high temperature could introduce unnecessary variation, meaning similar loan applications might receive different styles of analysis or recommendations.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [9]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [11]:
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002", "L006"]:
    prompt = f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    output = ask_llm(prompt)

    print(f"\n--- {letter_id} ---")
    print(output)

SUMMARY_PROMPT_V2 = """
You are an assistant to a microfinance loan officer.

Summarize the loan application in 3-4 sentences.
Be factual and neutral.
Do not invent any details.
"""

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]

    output = ask_llm(
        f"Summarize this loan application:\n\n{letter_text}",
        system_prompt=SUMMARY_PROMPT_V2,
        temperature=0
    )

    print(f"\n--- {letter_id} ---")
    print(output)


--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's struggling due to slow business, but expects things to improve after the festive season. He doesn't have collateral, but is hoping to repay the loan as soon as possible.

--- L006 ---
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He claims to be "business-minded" based on his friends' opinions, but has no prior experience. He promises to repay the loan in one year, once the businesses are successful, and offers no collateral, relying on his trustworthiness.

--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the

V1 produced useful summaries, but they were less controlled and slightly more interpretive. For example, in L002, V1 says Kwame is “struggling due to slow business” and that he “promises to repay the loan as soon as possible.” The original letter only says business has been slow and that he can pay back “whenever the money comes.” V2 stayed closer to the source by saying that his “business has been slow” and describing his request without strengthening his repayment commitment. For L006, V1 says Kofi will repay “once his businesses are successful,” while V2 more neutrally states that he expects the businesses to be successful. V2 was also consistently structured into about 4 factual sentences, making it more suitable for a loan officer.
“No invented details” is essential because a loan officer could make a financial decision based on information that the applicant never actually provided. Adding or changing details about income, repayment ability, collateral, or business experience could unfairly affect the applicant’s assessment. This failure mode is called hallucination, where an LLM generates information that is not supported by the source material.


### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [14]:
import json
import pandas as pd

EXTRACT_PROMPT = """
You are extracting structured information from a loan application letter.

Return ONLY a valid JSON object with EXACTLY these keys:

{
  "applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}

If a field is not stated in the letter, use null. Do not guess.

Example:

Letter:
"My name is Ama Mensah. I am requesting GHS 12,000 to purchase a new sewing machine.
My tailoring business makes a monthly profit of GHS 2,500. My sister has agreed to
serve as my guarantor. I plan to repay the loan within 8 months."

Output:
{
  "applicant_name": "Ama Mensah",
  "amount_ghs": 12000,
  "purpose": "purchase a new sewing machine",
  "monthly_profit_ghs": 2500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 8
}

Now extract the information from the following loan application.

Letter:
{letter_text}
"""


def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.replace("{letter_text}", letter_text)

    result = ask_llm(
        prompt,
        temperature=0
    )

    # Strip possible JSON code fences
    cleaned = result.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    # Convert JSON string to a Python dictionary
    try:
        return json.loads(cleaned)

    except json.JSONDecodeError:
        print("Warning: Could not parse the model output as JSON.")
        return None


results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)


df_extracted = pd.DataFrame(results)

df_extracted = df_extracted[
    [
        "letter_id",
        "applicant_name",
        "amount_ghs",
        "purpose",
        "monthly_profit_ghs",
        "has_collateral_or_guarantor",
        "repayment_months",
    ]
]

display(df_extracted)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**

The few-shot example should not come from the six letters being processed because that would expose the model to part of the dataset it is supposed to process independently. Using a separate example demonstrates the required format without giving the model an answer from the actual data.
“Use null, do not guess” is important because some applications do not provide every required field. For example, L002, L005, and L006 did not state a monthly profit, so the extraction correctly left monthly_profit_ghs missing rather than inventing a value. Without this instruction, the model could infer or fabricate a value from the surrounding information, which would make the structured data unreliable.
Temperature 0 is appropriate for extraction because we want the same factual information to be extracted consistently and predictably each time. Creativity is not useful when copying facts into structured fields. In creative tasks, however, a higher temperature can be useful because variation and originality may be desirable.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.